In [13]:

"""
Comparing images using ORB/SIFT feature detectors
and structural similarity index. 

"""

from PIL import Image, ImageChops 
from skimage.metrics import structural_similarity
import cv2
import os
import glob
import tifffile # pip install tifffile
import numpy as np
from pathlib import Path

# Find all orthophoto .tif files
tif_files_myversion = list(Path('/home/hubert/oats/results/3_0_0/').glob('**/odm_orthophoto.tif'))
tif_files_latest   = list(Path('/home/hubert/oats/results/3_5_6/').glob('**/odm_orthophoto.tif'))

# Create dictionaries keyed by relative path
myversion_dict = {os.path.relpath(str(f), '/home/hubert/oats/results/3_0_0/'): str(f) for f in tif_files_myversion}
latest_dict    = {os.path.relpath(str(f), '/home/hubert/oats/results/3_5_6/'): str(f) for f in tif_files_latest}

# Find common files
common_keys = sorted(set(myversion_dict.keys()) & set(latest_dict.keys()))
print("Common files:", common_keys)
print("Number of common files:", len(common_keys))






base_latest = "/home/hubert/oats/results/3_0_0/"
base_myversion = "/home/hubert/oats/results/3_5_6/"

 #Works well with images of different dimensions
def orb_sim(img1, img2):
      try: 
          # SIFT is no longer available in cv2 so using ORB 
          orb = cv2.ORB_create()
        
          # detect keypoints and descriptors
          kp_a, desc_a = orb.detectAndCompute(img1, None)
          kp_b, desc_b = orb.detectAndCompute(img2, None)
        
          # define the bruteforce matcher object
          bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
            
          #perform matches. 
          matches = bf.match(desc_a, desc_b)
          #Look for similar regions with distance < 50. Goes from 0 to 100 so pick a number between.
          similar_regions = [i for i in matches if i.distance < 50]  
          if len(matches) == 0:
            return 0
          return len(similar_regions) / len(matches)

      except Exception as e:
          print('there is an error', e)

#Needs images to be same dimensions
def structural_sim(img1, img2):

  sim, diff = structural_similarity(img1, img2, full=True, data_range=1.0)
  return sim


def get_8bit_tif(path):
    img = tifffile.imread(path)

    # If RGBA, drop everything beyong 3 channels
    if len(img.shape) == 3:
        img = img[:, :, :3]
   

    # Convert RGB → Grayscale                                 this is a problem with non rgb so we need to hand multiband
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX)
    return img.astype(np.uint8)


for key in common_keys:

  
    
    path_latest = os.path.join(base_latest, key)
    path_myversion = os.path.join(base_myversion, key)

    print(key)


    img1 = get_8bit_tif(path_latest)
    img2 = get_8bit_tif(path_myversion)

   

        
    
    #Resize for SSIM                              upload to git raster
    from skimage.transform import resize
    img3 = resize(
        img2,
        (img1.shape[0], img1.shape[1]),
        anti_aliasing=True,
        preserve_range=True
    ).astype(np.uint8)



        
    difference_o = ImageChops.difference(
    Image.fromarray(img1),
    Image.fromarray(img3)
    )
    difference_o.show()
    orb_similarity = orb_sim(img1, img3)  #1.0 means identical. Lower = not similar
    
    ssim_val = structural_sim(img1, img3)
    print("Similarity using SSIM is: ", ssim_val)
    print("Similarity using ORB is: ", orb_similarity)    # orb printed was previosly places=d in the loop so it was triggering an error

    




Common files: ['datasets/brighton/odm_orthophoto/odm_orthophoto.tif', 'datasets/brighton_masked/odm_orthophoto/odm_orthophoto.tif', 'datasets/brighton_no_exif/odm_orthophoto/odm_orthophoto.tif', 'datasets/brighton_spaces/odm_orthophoto/odm_orthophoto.tif']
Number of common files: 4
datasets/brighton/odm_orthophoto/odm_orthophoto.tif
Similarity using SSIM is:  0.23489210038666175
Similarity using ORB is:  0.5384615384615384
datasets/brighton_masked/odm_orthophoto/odm_orthophoto.tif
Similarity using SSIM is:  0.24672185003554867
Similarity using ORB is:  0.593607305936073
datasets/brighton_no_exif/odm_orthophoto/odm_orthophoto.tif
Similarity using SSIM is:  0.06525904068188534
Similarity using ORB is:  0.07692307692307693
datasets/brighton_spaces/odm_orthophoto/odm_orthophoto.tif



(eog:959823): EOG-WARNING **: 20:21:02.666: Generating thumbnail failed: Child process exited with code 1

(eog:959823): EOG-WARNING **: 20:21:02.667: Thumbnail creation failed

(eog:959823): EOG-WARNING **: 20:21:03.276: Thumbnail creation failed

(eog:959823): EOG-WARNING **: 20:21:03.664: Generating thumbnail failed: Child process exited with code 1

(eog:959823): EOG-WARNING **: 20:21:03.664: Thumbnail creation failed

(eog:959823): EOG-WARNING **: 20:21:03.665: Thumbnail creation failed

(eog:959823): EOG-WARNING **: 20:21:03.851: Thumbnail creation failed

(eog:959823): EOG-WARNING **: 20:21:03.851: Thumbnail creation failed

(eog:959823): EOG-WARNING **: 20:21:03.852: Thumbnail creation failed


Similarity using SSIM is:  0.37074940944800494
Similarity using ORB is:  0.6369047619047619



(eog:959823): EOG-WARNING **: 20:21:04.448: Thumbnail creation failed
